In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import roc_auc_score

folder_name = "borenstein-lab microbiome-metabolome-curated-data main data-processed_data_JACOBS_IBD_FAMILIES_2016"

metadata = pd.read_csv(f"{folder_name}/metadata.tsv", sep="\t")
mtb = pd.read_csv(f"{folder_name}/mtb.tsv", sep="\t")

feature_cols = [c for c in mtb.columns if c != "Sample"]
X = metadata.merge(mtb, on="Sample")[feature_cols]
y = metadata["Study.Group"]

print("X shape:", X.shape)
print(y.value_counts())

X shape: (90, 4626)
Study.Group
Normal    54
CD        26
UC        10
Name: count, dtype: int64


In [3]:
# Same sparsity filter and outlier detection as before, for consistency across methodologies
sparsity = (X == 0).sum(axis=0) / len(X) * 100
X_filtered = X[sparsity[sparsity <= 80].index]

iso = IsolationForest(contamination=0.05, random_state=42)
outlier_flags = iso.fit_predict(X_filtered)
X_clean = X_filtered[outlier_flags == 1].reset_index(drop=True)
y_clean = y[outlier_flags == 1].reset_index(drop=True)

print("Samples after cleaning:", len(X_clean))

# SIAMCAT preprocessing: prevalence filter (>=10%) + log-unit normalization
prevalence = (X_clean > 0).mean(axis=0)
keep_siamcat = prevalence[prevalence >= 0.10].index
feat_log = np.log10(X_clean[keep_siamcat] + 1e-6)
feat_norm = feat_log.div(feat_log.sum(axis=1), axis=0)

print("Features after SIAMCAT prevalence filter:", feat_norm.shape[1])

Samples after cleaning: 85
Features after SIAMCAT prevalence filter: 2342


In [4]:
y_binary = y_clean.map(lambda g: "Normal" if g == "Normal" else "IBD")
le_bin = LabelEncoder()
y_bin_enc = le_bin.fit_transform(y_binary)

train_idx, test_idx = train_test_split(np.arange(len(y_bin_enc)), test_size=0.2, stratify=y_bin_enc, random_state=42)
y_train, y_test = y_bin_enc[train_idx], y_bin_enc[test_idx]

def auc_feature_ranking(Xd, yd, n_features=100):
    scores = {}
    for col in Xd.columns:
        try:
            score = roc_auc_score(yd, Xd[col])
            scores[col] = abs(score - 0.5)
        except ValueError:
            scores[col] = 0
    ranked = sorted(scores, key=scores.get, reverse=True)
    return ranked[:n_features]

# Rank using only training data to avoid leakage
top_n = min(100, feat_norm.shape[1])
top_features = auc_feature_ranking(feat_norm.iloc[train_idx], y_train, n_features=top_n)

X_train_s = feat_norm[top_features].iloc[train_idx]
X_test_s = feat_norm[top_features].iloc[test_idx]

print("Features selected:", len(top_features))
print("Train shape:", X_train_s.shape, "Test shape:", X_test_s.shape)

Features selected: 100
Train shape: (68, 100) Test shape: (17, 100)


In [5]:
# Random Forest
rf_siamcat = RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1)
rf_siamcat.fit(X_train_s, y_train)
rf_probs = rf_siamcat.predict_proba(X_test_s)[:, 1]
rf_auc = roc_auc_score(y_test, rf_probs)

# LASSO
lasso = LogisticRegressionCV(penalty="l1", solver="liblinear", cv=5, max_iter=5000, random_state=42)
lasso.fit(X_train_s, y_train)
lasso_probs = lasso.predict_proba(X_test_s)[:, 1]
lasso_auc = roc_auc_score(y_test, lasso_probs)

# Elastic Net
enet = LogisticRegression(penalty="elasticnet", solver="saga", max_iter=20000, random_state=42)
enet_grid = GridSearchCV(enet, {"C": [0.01, 0.1, 1, 10], "l1_ratio": [0.1, 0.5, 0.9]}, cv=5, scoring="roc_auc")
enet_grid.fit(X_train_s, y_train)
enet_probs = enet_grid.predict_proba(X_test_s)[:, 1]
enet_auc = roc_auc_score(y_test, enet_probs)

# Ridge
ridge = LogisticRegression(penalty="l2", solver="lbfgs", max_iter=20000, random_state=42)
ridge_grid = GridSearchCV(ridge, {"C": [0.01, 0.1, 1, 10]}, cv=5, scoring="roc_auc")
ridge_grid.fit(X_train_s, y_train)
ridge_probs = ridge_grid.predict_proba(X_test_s)[:, 1]
ridge_auc = roc_auc_score(y_test, ridge_probs)

print(f"SIAMCAT-style Random Forest — AUC: {rf_auc:.3f}")
print(f"SIAMCAT-style LASSO — AUC: {lasso_auc:.3f}")
print(f"SIAMCAT-style Elastic Net — AUC: {enet_auc:.3f}")
print(f"SIAMCAT-style Ridge — AUC: {ridge_auc:.3f}")

SIAMCAT-style Random Forest — AUC: 0.857
SIAMCAT-style LASSO — AUC: 0.871
SIAMCAT-style Elastic Net — AUC: 0.786
SIAMCAT-style Ridge — AUC: 0.800
